# Experiment 4: transaction cost model

Strategy: the configured DCC-GARCH model with the 60-month calibration rescale (`config/risk_config.json`), 10% band, primary cost 2 bps per traded notional (`primary_cost_bps`), 1976-2014, data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`). Results below read the experiment 1 output in `reports/dcc_garch_studies/band/`, so run that notebook first. Roll cost is experiment 2 and collateral income is experiment 3; this notebook builds on both and does not repeat them.

## Goal

Replace the flat 2 bps with a cost model that separates what the broker and exchange charge (explicit) from what trading does to the price you get (implicit), calibrate it from the liquidity data we have, and stress it. The earlier deck only ran a flat 0/5/10 bps sensitivity. Judge every change by net Sharpe and CAGR, cost drag in bps a year, turnover, the breakeven cost per unit traded, and cost by asset class.

## Current cost model

**What we are doing.** Charge one aggregate rate on every unit of traded notional. The rate is a proxy for everything: commission, spread, impact and slippage are not separated.

**Inputs**
- Executed positions this month and last month, as fractions of NAV.
- `primary_cost_bps` = 2, applied to every market alike. Scenarios 0, 1, 2 and 5 bps.
- Optional scheduled roll fee on matched continuing notional (experiment 2; off in production).

**How we process them** (`src/transaction_cost_model/`)
1. `traded_exposure` takes `|w_new - w_old|` per market. A source switch closes the old and opens the new position, so it counts both legs.
2. `LinearBpsCost` multiplies by `bps / 10,000`. The total is deducted from the month's NAV (`backtest/engine.py`).
3. Terminal liquidation is charged once at the end of the sample.
4. The executed-limit step (`trade/feasibility.py`) prices the trades, so cost reduces room under the gross cap and risk ceiling.

*Example:* a market moving from 3% to 5% of NAV trades 2% of NAV and pays 2% x 2 bps = 0.04 bps of NAV. A sign flip from -3% to +3% trades 6% and pays three times as much.

**Output.** A monthly cost as a fraction of NAV, charged to the ledger and split by market.

## What the target model covers

| | Cost | Depends on | Where it shows up |
|---|---|---|---|
| **Explicit** | Commission and brokerage | Contracts traded (a fixed amount per contract, so bps of notional falls as price rises) | Per-contract fee / contract notional |
| | Exchange, clearing and regulatory fees | Contracts traded | Same form |
| | Financing and margin | Capital tied up, not trades | Experiment 3 (collateral income); margin schedule unavailable |
| **Implicit** | Half the bid-ask spread | Market, liquidity regime, volatility | Every trade, linear in size |
| | Market impact | Trade size against volume, volatility | Convex in size: matters at scale |
| | Slippage and delay | Gap between decision price and fill | Monthly returns assume execution at the reference price; any gap is unmodelled |
| | Roll | Calendar and roll liquidity | Experiment 2 |
| | Rounding to whole contracts | Contract notional against position | Tracking error, not a fee |

## Implications

- One rate for all markets cannot be right for all of them. Cost per unit traded varies across asset class and over time with liquidity and volatility.
- The flat rate is linear in size, so it cannot show the capacity limit: impact is convex, and the strategy's break-even AUM depends on it.
- The decisive question is how large costs would have to be to matter. That is the breakeven below.

## Results and interpretation

Gross return, turnover and cost are annualised means over 1976-2014 from the experiment 1 ledger (TREND, 10% band, 2 bps).


In [1]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
BAND = ROOT / "reports/dcc_garch_studies/band"  # experiment 1 output; run that notebook first
pd.options.display.float_format = "{:.4f}".format

from src.config import ASSET_MAP_PATH, RAW_DATA_DIR

ledger = pd.read_csv(BAND / "correlated_band10_2bps_trend_ledger.csv", index_col=0)
by_market = pd.read_csv(BAND / "correlated_band10_2bps_trend_cost.csv", index_col=0)
categories = pd.read_csv(BAND / "turnover_breakdown.csv").query("band == 0.1 and cost_bps == 2 and strategy == 'trend'")
asset_class = pd.read_csv(ASSET_MAP_PATH, index_col=0).AssetClass
print(f"{len(ledger)} months, {ledger.index[0]} to {ledger.index[-1]}")

468 months, 1976-01 to 2014-12


In [2]:
gross, turnover, cost = (ledger[c].mean() * 12 for c in ("gross_return", "turnover", "cost"))
pd.Series(
    {
        "gross return (a year)": gross,
        "turnover (x NAV a year)": turnover,
        "cost at 2 bps (a year)": cost,
        "cost / gross return": cost / gross,
        "breakeven all-in cost (bps per unit traded)": gross / turnover * 1e4,
        "return on execution costs (gross / cost)": gross / cost,
    }
).to_frame("trend")

,trend
gross return (a year),0.1124
turnover (x NAV a year),8.6700
cost at 2 bps (a year),0.0017
cost / gross return,0.0154
breakeven all-in cost (bps per unit traded),129.6819
return on execution costs (gross / cost),64.8409


- **At 2 bps the cost is small:** 17 bps a year against 11.2% gross, 1.5% of gross return. Gross return is 65 times execution cost, and the breakeven is about 130 bps per unit traded, 65 times the assumption.
- **Turnover is 8.7 times NAV a year**, so the cost is a small rate on a large volume. Any cost that scales with the square of size, or that rises in turbulence, hits the volume hardest.

In [3]:
classes = by_market.sum().mul(12 / len(by_market)).groupby(asset_class).sum().mul(1e4)
classes.rename("cost_bps_ann").to_frame().assign(share=classes / classes.sum())

,cost_bps_ann,share
AssetClass,,
COMM,7.4116,0.4274
EQ,0.9018,0.0520
FI,4.0859,0.2356
FX,4.9406,0.2849


In [4]:
mix = categories.set_index("category").turnover_ann
mix.to_frame().assign(share=mix / mix.sum())

,turnover_ann,share
category,,
source_switches,0.0016,0.0002
entries,1.0817,0.1248
exits,0.8734,0.1007
reversals,4.7820,0.5516
resizes,1.8578,0.2143
terminal_liquidation,0.0736,0.0085


- **Commodities pay 43% of the cost, FX 28%, bonds 24%, equities 5%.** The rate is the same everywhere, so this is only the turnover mix. It is the baseline against which a per-class rate has to be read.
- **Sign reversals are 55% of turnover** (4.8 of 8.7 times NAV), resizes 21%, entries and exits 23%. A reversal trades both legs, so it costs twice a same-size entry. The band already skips small resizes; it cannot skip a flip.

In [5]:
rows = {}
for bps in (0, 2, 5, 10, 20, 50, 100):
    net = ledger.gross_return - ledger.turnover * bps / 1e4  # first order: positions held at the 2 bps path
    rows[bps] = {"mean_ann": net.mean() * 12, "sharpe": net.mean() / net.std() * np.sqrt(12)}
pd.DataFrame(rows).T.rename_axis("one-way bps")

,mean_ann,sharpe
one-way bps,,
0,0.1124,1.2176
2,0.1107,1.1991
5,0.1081,1.1712
10,0.1038,1.1247
20,0.0951,1.0314
50,0.0691,0.7494
100,0.0257,0.2777


- **Costs have to rise a long way.** Net Sharpe falls from 1.22 (0 bps) to 1.20 (2), 1.12 (10), 1.03 (20), 0.75 (50) and 0.28 (100). A 10-fold error in the rate (20 bps) costs 0.17 Sharpe; the strategy is not cost-fragile at these levels.
- **First-order only.** Positions are held at the 2 bps path. At higher cost the band, the executed limits and the reversal share would change, so the real curve is flatter or steeper than this. That is C0.
- **Single in-sample path.** The gaps are untested.

In [6]:
K = 3 - 2 * np.sqrt(2)


def corwin_schultz(bars):
    """AFML 19.3.4 on daily bars; negative alphas set to zero as the paper recommends."""
    h, l = np.log(bars.High), np.log(bars.Low)
    beta = (h - l) ** 2 + ((h - l) ** 2).shift(1)
    gamma = (np.maximum(h, h.shift(1)) - np.minimum(l, l.shift(1))) ** 2
    alpha = ((np.sqrt(2 * beta) - np.sqrt(beta)) / K - np.sqrt(gamma / K)).clip(lower=0)
    return 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))


def roll_spread(bars):
    """AFML 19.3.2: 2c with c = sqrt(max(0, -cov(dp_t, dp_t-1))), relative to the mean price."""
    dp = bars.Close.diff().dropna()
    return 2 * np.sqrt(max(0.0, -dp.cov(dp.shift(1)))) / bars.Close.mean()


archive = zipfile.ZipFile(RAW_DATA_DIR / "FuturesUnderlyingData.zip")  # read only; raw data stays untouched
proxies = {}
for name in archive.namelist():
    if not name.endswith(".csv") or "/" in name:
        continue
    bars = pd.read_csv(archive.open(name), index_col=0, parse_dates=True)
    bars = bars[(bars.High >= bars.Low) & (bars.Low > 0)]
    proxies[name[:-4]] = {
        "corwin_schultz_bps": corwin_schultz(bars).median() * 1e4,
        "roll_bps": roll_spread(bars) * 1e4,
        "zero_volume_days": (bars.Volume == 0).mean(),
    }
proxies = pd.DataFrame(proxies).T.assign(asset_class=asset_class)
proxies.groupby("asset_class").median().join(classes.rename("realised_cost_bps_ann"))

,corwin_schultz_bps,roll_bps,zero_volume_days,realised_cost_bps_ann
asset_class,,,,
COMM,5.3945,0.0000,0.0002,7.4116
EQ,14.9935,52.9496,0.0036,0.9018
FI,3.6147,0.0000,0.0040,4.0859
FX,0.1468,2.9921,0.0003,4.9406


- **Neither spread proxy works as a level.** Corwin-Schultz gives a median of 15 bps for equity index futures and 0.1 bp for FX, against 2 bps all-in; the ordering (equities widest) is the reverse of what liquid index futures should show. Roll gives 0 for commodities and bonds, where daily changes are positively autocorrelated (the formula floors a positive covariance at zero), and 53 bps for equities.
- **Likely cause:** the daily high-low range includes volatility and session effects the estimator cannot remove, and the close-to-close covariance is dominated by trend and non-synchronous closes. This is an inference; there are no quotes in the data to check against.
- **Volume is usable only for relative liquidity.** Zero-volume days are under 0.5% in every class. Volume is in reported-series units, so it cannot be turned into dollars without point values (experiment 3).
- **Consequence:** the spread level has to come from outside the data (tick sizes, broker quotes) and the daily bars supply only the time variation. C3 below therefore fixes the level at 1, 2 and 5 bps and tests only how the cost moves with volatility.

## AFML improvements

Read: Ch. 2 (2.4.1), Ch. 11 (11.2), Ch. 14 (14.6, 14.7.3), Ch. 19 (19.3-19.4), Ch. 21 (21.3), Ch. 10 (10.4-10.5).

| Idea | What the book says | Use here |
|---|---|---|
| Costs are hard to simulate (11.2, item 5) | The only way to be certain of a trading cost is to trade against the book. | Report costs as ranges and as breakevens, never as a point estimate. |
| Rebalance cost and bid-ask cost in the ETF trick (2.4.1) | Per instrument `tau_i` (e.g. 1 bp) per $1 traded. Rebalance cost `c_t = sum (|h_{t-1}| p_t + |h_t| o_{t+1}) phi_t tau_i` is a negative dividend, not part of the NAV series. A separate bid-ask cost `c~_t = sum |h_{t-1}| p_t phi_t tau_i` is charged when a unit is bought or sold. Costs "are not necessarily linear" and can be simulated from these inputs. | Move from one `tau` to a per-market `tau_i` in contracts, using the point value `phi`. Keep the cost out of the return series and charge it in the ledger, as now. |
| Volume limits the basket (2.4.1) | Tradeable units are set by the least active member: `v_t = min_i(v_{i,t} / |h_{i,t-1}|)`. | A participation cap per market: trade size against reported volume. Volume is in reported-series units (experiment 3). |
| Implementation shortfall statistics (14.6) | Broker fees per turnover, average slippage per turnover, dollar performance per turnover, return on execution costs ("a large multiple, to ensure the strategy survives worse-than-expected execution"). | Report all four. Two are already computed above: 65x return on execution costs and a 130 bps breakeven. Fees and slippage need the explicit/implicit split. |
| Roll model (19.3.2) | `c = sqrt(max(0, -cov(dp_t, dp_{t-1})))`; effective spread `2c`. Useful where quotes are unrepresentative. | Spread proxy from daily closes. Tested below: unusable as a level. |
| Corwin-Schultz (19.3.4) | Spread from the high-low ratio over two bars, negative alphas set to zero. | Spread proxy from daily highs and lows. Tested below: unusable as a level. |
| Kyle, Amihud, Hasbrouck lambdas (19.4.1-19.4.3) | Price change per unit of signed flow (Kyle), per dollar of volume (Amihud), or per square root of dollar volume (Hasbrouck). The book reports that daily Amihud estimates rank-correlate with intraday effective spread, and that t-values are more informative than the point estimates. | Impact slope from daily `|return|` on dollar volume (Amihud form) is the only one the daily bars allow. It needs the contract multiplier for dollar volume, which is missing. |
| Transaction cost functions in the optimiser (21.3) | `tau_h[omega] = sum c_{n,h} sqrt(|omega_{n,h} - omega_{n,h-1}|)`, with an asset-specific, time-varying `c_{n,h}`. | The functional form for impact with time-varying scales. The book's form is concave in total cost; the usual square-root law has cost per unit traded growing as the square root of size, so total cost grows as size to the 1.5. I would use the second and treat that as my reading, not the book's. |
| Discretising bets (10.4-10.5) | Rounding sizes to steps cuts turnover from small resizes. | Whole-contract rounding is a real cost at small AUM: it adds tracking error and removes small trades. |
| Deflated Sharpe (14.7.3) | Judge a Sharpe against the maximum expected from the number of trials. | Cost scenarios are not trials of the strategy, but any cost-aware change to the band or the signal is. |


## Experiments kept

| # | Change | Experiment | Needs | Effort |
|---|---|---|---|---|
| C0 | Stress grid: 0, 2, 5, 10, 20, 50, 100 bps; rerun through the engine, not first order | Net Sharpe, CAGR, turnover, limit overrides, breakeven. The first-order table above holds positions fixed; the band and limits react to cost | Nothing new | Low |
| C3 | Spread that moves with volatility | Half-spread = `h x sigma_{i,t} / sigma_{i,median}`, sigma from the risk model. Charge higher in turbulence, when trend books trade most | One level `h` for all markets (1, 2, 5 bps); no per-market level is available | Low |
| C4 | Convex impact | `Y sigma_{i,d} sqrt(Q / ADV)` per unit traded, `Q` from AUM x `|dw|`, ADV from reported volume x price x point value. Find the AUM at which Sharpe falls by 0.1 and the AUM at which net return is zero | Point values, AUM grid, `Y` from literature (not fitted here) | Medium |
| C5 | Slippage from the fill delay | Decision price at month end, fill at the next open. Measure the gap between close-to-open return on trade days and on others, signed by trade direction | Daily Open and Close (available) | Medium |

**Not recommended:** fitting Kyle or Hasbrouck lambdas. They need signed trades or intraday data, and the daily bars have neither.

**Not run here:** per-class rates, explicit per-contract fees, proxy validation, cost-aware sign flips and the 14.6 statistics. They need outside data or build on the four tests above.

## Tests

Logic lives in `scripts/run_transaction_costs.py` and its parameters in `config/transaction_costs.json`. The script reruns the experiment 1 strategy (TREND, 10% band) and writes to `reports/dcc_garch_studies/transaction_costs/`. This notebook runs it if that folder is missing (a few minutes, mostly the DCC-GARCH fit) and reads the output.

**Method.** C0 reruns the engine at each cost. C3, C4 and C5 price the trades of the 2 bps path with the richer cost and subtract it from gross return (first order: positions are not re-optimised). C0 shows how large that approximation is.

**Assumptions I made, not taken from the data.**
- *Point values (C4):* standard contract sizes written from memory, with non-USD contracts at rough end-2014 exchange rates. They are not checked against exchange specifications, and the supplied prices are adjusted series (for example crude oil, natural gas and the Russell 2000 do not sit at their real December 2014 levels), so contract values are indicative. The table is below.
- *Impact coefficient:* `Y` = 0.5 and 1.0 (the usual range for the square-root law). Not fitted.
- *AUM grid:* $10m, $100m, $1bn, $10bn.
- *Sensitivity:* ADV is also scaled by 0.5 and 2 so that an error in the point values moves the result inside a stated range.
- *Costs add up:* impact (C4) and slippage (C5) are charged on top of the 2 bps flat rate, which stands for commission and spread. In the combined test the flat rate is replaced by the C3 spread.

In [7]:
from run_transaction_costs import main

REFERENCE = ROOT / "reports/cost_revision_20261026"  # hashed inputs (data only)
TESTS = ROOT / "reports/dcc_garch_studies/transaction_costs"
if not TESTS.exists():
    main(REFERENCE, TESTS)

### C0: stress grid through the engine

**What we are doing.** Charge each flat cost from 0 to 100 bps and let the strategy react: the band, the executed limits and the sign-flip share all see the cost.

**Inputs.** The experiment 1 targets and DCC-GARCH covariance, `LinearBpsCost(bps)`, 10% band.

**How we process them.** `simulate` with `BandPolicy` at each cost. The first-order columns reprice the 2 bps path (positions fixed). The breakeven is found by bisection on the engine.

**Output.** Net Sharpe and CAGR, turnover, cost, months where the limits cut the position, resizes the band skipped, and the cost at which net return reaches zero.

In [8]:
grid = pd.read_csv(TESTS / "c0_stress_grid.csv").set_index("bps")
grid["sharpe_gap"] = grid.sharpe - grid.first_order_sharpe
grid

,mean_ann,sharpe,cagr,turnover_ann,cost_ann,limit_override_months,skipped_resizes,first_order_sharpe,first_order_cagr,sharpe_gap
bps,,,,,,,,,,
0,0.1124,1.2176,0.1137,8.6691,0.0000,121,5943,1.2176,0.1137,-0.0000
2,0.1107,1.1991,0.1118,8.6700,0.0017,122,5950,1.1991,0.1118,0.0000
5,0.1081,1.1713,0.1090,8.6713,0.0043,123,5949,1.1712,0.1090,0.0001
10,0.1038,1.1249,0.1042,8.6735,0.0087,123,5962,1.1247,0.1042,0.0001
20,0.0951,1.0316,0.0948,8.6780,0.0174,127,5970,1.0314,0.0948,0.0002
50,0.0690,0.7488,0.0667,8.6921,0.0435,139,5980,0.7494,0.0668,-0.0005
100,0.0252,0.2724,0.0211,8.7186,0.0872,159,6033,0.2777,0.0217,-0.0053


In [9]:
pd.read_csv(TESTS / "c0_breakevens.csv", index_col=0)

,bps per unit traded
net mean return reaches zero (engine),128.5659
net Sharpe falls 0.1 (engine),12.7673
net mean return reaches zero (first order),129.6819


- **First order is accurate here.** The engine and the fixed-position repricing agree to 0.001 Sharpe up to 20 bps and to 0.005 at 100 bps (0.272 against 0.278). Turnover barely moves (8.67 to 8.72 times NAV), so the strategy does not trade less when trading is dear. That also means the cost rises one for one and the earlier table stands.
- **The limits react more than the band does.** Limit-override months rise from 122 at 2 bps to 159 at 100 bps (of 468), and skipped resizes from 5,950 to 6,033. Higher costs eat room under the gross cap and risk ceiling, but the effect on return is small.
- **Net return reaches zero at 129 bps per unit traded** (engine), against 130 from the first-order formula. Sharpe falls by 0.1 at 12.8 bps. CAGR goes from 11.2% at 2 bps to 2.1% at 100 bps.

### C3: spread that moves with volatility

**What we are doing.** Make the half-spread proportional to the market's own volatility relative to its median, so trades in turbulent months cost more.

**Inputs.** Traded notional per market-month from the 2 bps path (including source switches and terminal liquidation); the DCC-GARCH volatility forecast known at each trade date; one half-spread level `h` for all markets.

**How we process them.** `multiplier = sigma_{i,t} / median(sigma_i)` over the market's listed months. Cost = `h x multiplier x traded`. A zero forecast after a market's first one (RL, 2009-01, the source switch) is carried forward from the prior month. The last case holds total cost equal to the flat 2 bps, so any Sharpe difference comes from timing alone.

**Output.** Annual cost, effective bps per unit traded, net Sharpe and CAGR against the flat rate, and the multiplier by class and in the most volatile months.

In [10]:
pd.read_csv(TESTS / "c3_spread.csv")

,case,spread,cost_ann_bps,effective_bps,mean_ann,sharpe,cagr
0,1 bps,flat,8.6700,1.0000,0.1116,1.2084,0.1128
1,1 bps,volatility-scaled,8.9944,1.0374,0.1115,1.2080,0.1128
2,2 bps,flat,17.3400,2.0000,0.1107,1.1991,0.1118
3,2 bps,volatility-scaled,17.9887,2.0748,0.1106,1.1984,0.1118
4,5 bps,flat,43.3499,5.0000,0.1081,1.1712,0.1090
5,5 bps,volatility-scaled,44.9718,5.1871,0.1079,1.1694,0.1088
6,1.93 bps scaled vs 2 bps flat,flat,17.3400,2.0000,0.1107,1.1991,0.1118
7,1.93 bps scaled vs 2 bps flat,volatility-scaled,17.3400,2.0000,0.1107,1.1990,0.1118


In [11]:
pd.read_csv(TESTS / "c3_regimes.csv", index_col=0)

,value
"multiplier, all months",1.0374
"multiplier, highest-volatility months",1.6119
share of turnover in highest-volatility months,0.0698
"multiplier, COMM",1.0446
"multiplier, EQ",1.1115
"multiplier, FI",1.0319
"multiplier, FX",1.0177


- **Linking the spread to volatility adds about 4% to cost.** At `h` = 2 bps the effective rate is 2.07 bps and annual cost 18.0 bps against 17.3. Net Sharpe moves from 1.199 to 1.198 (1.171 to 1.169 at 5 bps).
- **The trades are not in the turbulent months.** The most volatile 10% of months (by traded-weighted multiplier) have a multiplier of 1.61 but carry only 7% of turnover. With total cost held at 2 bps, Sharpe is 1.1991 against 1.1990: timing alone costs nothing measurable.
- **Equities run highest (1.11)** and FX lowest (1.02). The spread-volatility link is an assumption (proportional, one level for all markets). A link steeper than proportional in a crisis would bite more, and this test cannot tell, because the sample has no quotes.

### C4: convex market impact and capacity

**What we are doing.** Charge impact that grows with the square root of the trade relative to daily volume, so cost per unit traded rises with AUM and total cost grows as AUM to the 1.5.

**Inputs.** Traded notional per market-month (fraction of NAV); AUM; the forecast volatility; ADV in dollars = 60-day mean reported volume at the trade date x endpoint close x point value; `Y` = 0.5 and 1.0.

**How we process them.** Per unit traded, `Y x sigma_daily x sqrt(Q / ADV)` with `Q` = AUM x traded and `sigma_daily` = annual forecast / sqrt(252). The whole month's trade is assumed to execute on the trade date. The charge is added to the flat 2 bps. AUM is searched on a log scale for the point where Sharpe falls by 0.1 and where net mean return reaches zero.

**Output.** Impact cost, net Sharpe and CAGR by AUM; capacity under each `Y` and ADV scale; impact by period and market.

In [12]:
values = pd.read_csv(TESTS / "c4_contract_values.csv", index_col=0)
values.assign(asset_class=asset_class).groupby("asset_class").contract_notional_dec2014.describe().round(0)

,count,mean,std,min,25%,50%,75%,max
asset_class,,,,,,,,
COMM,26.0000,94385.0000,151162.0000,14353.0000,28889.0000,66752.0000,87151.0000,801630.0000
EQ,14.0000,197832.0000,201659.0000,15204.0000,52867.0000,112664.0000,301943.0000,653285.0000
FI,10.0000,207224.0000,199185.0000,106193.0000,124233.0000,144294.0000,160971.0000,766717.0000
FX,8.0000,216325.0000,303725.0000,21617.0000,89441.0000,108860.0000,175054.0000,957139.0000


In [13]:
impact = pd.read_csv(TESTS / "c4_impact.csv")
impact.query("adv_scale == 1.0")[
    ["coefficient", "aum", "impact_ann_bps", "impact_bps_per_unit", "trades_above_10pct_adv", "mean_ann", "sharpe", "cagr"]
].set_index(["coefficient", "aum"])

impact_ann_bps  impact_bps_per_unit  \
coefficient aum                                                     
0.5000      10000000.0000            25.4885               2.9399   
            100000000.0000           80.6018               9.2966   
            1000000000.0000         254.8853              29.3986   
            10000000000.0000        806.0180              92.9664   
1.0000      10000000.0000            50.9771               5.8797   
            100000000.0000          161.2036              18.5933   
            1000000000.0000         509.7705              58.7971   
            10000000000.0000       1612.0360             185.9329   

                              trades_above_10pct_adv  mean_ann  sharpe    cagr  
coefficient aum                                                                 
0.5000      10000000.0000                     0.0028    0.1082  1.1720  0.1090  
            100000000.0000                    0.0333    0.1026  1.1127  0.1030  
            1000000000.0000                   0.1650    0.0852  0.9189  0.0840  
            10000000000.0000                  0.4215    0.0301  0.3017  0.0254  
1.0000      10000000.0000                     0.0028    0.1056  1.1447  0.1062  
            100000000.0000                    0.0333    0.0946  1.0240  0.0942  
            1000000000.0000                   0.1650    0.0597  0.6287  0.0566  
            10000000000.0000                  0.4215   -0.0505 -0.4143 -0.0570

In [14]:
pd.read_csv(TESTS / "c4_capacity.csv").set_index(["coefficient", "adv_scale"]).round(-6)

,,aum_sharpe_falls_by_tolerance,aum_net_return_zero
coefficient,adv_scale,,
0.5000,0.5000,67000000.0000,9431000000.0000
1.0000,0.5000,17000000.0000,2358000000.0000
0.5000,1.0000,133000000.0000,18863000000.0000
1.0000,1.0000,33000000.0000,4716000000.0000
0.5000,2.0000,267000000.0000,37726000000.0000
1.0000,2.0000,67000000.0000,9431000000.0000


In [15]:
pd.read_csv(TESTS / "c4_by_period.csv").set_index("period")

,impact_ann_bps,impact_bps_per_unit
period,,
1976-1984,452.1799,65.6610
1985-1994,312.4644,34.4892
1995-2004,173.9954,18.4085
2005-2014,100.6308,11.0543


In [16]:
by_market = pd.read_csv(TESTS / "c4_by_market.csv", index_col=0)
share = by_market.groupby("asset_class").impact_bps_ann.sum()
print((share / share.sum()).round(3).to_string())
by_market.head(8)

asset_class
COMM   0.8040
EQ     0.0340
FI     0.0270
FX     0.1350


,asset_class,impact_bps_ann
JO,COMM,39.9815
ZP,COMM,31.3282
ZM,COMM,21.4434
ZO,COMM,17.2243
ZA,COMM,17.0098
CN,FX,11.2138
MW,COMM,9.5922
ZF,COMM,8.2344


- **Impact is what limits size, not the flat rate.** With `Y` = 0.5 and the table's point values, impact costs 25 bps a year at $10m, 81 at $100m, 255 at $1bn and 806 at $10bn, on top of 17 bps of flat cost. Net Sharpe is 1.17, 1.11, 0.92 and 0.30 against 1.20.
- **Capacity.** Sharpe falls by 0.1 at about $133m (`Y` = 0.5) or $33m (`Y` = 1.0), and net return reaches zero at about $19bn or $4.7bn. Across `Y` and the ADV scale, the Sharpe point runs from $17m to $267m and the zero-return point from $2.4bn to $38bn. Read these as orders of magnitude: the point values are unverified and `Y` is not fitted.
- **The early sample drives the number.** Impact per unit traded at $1bn is 66 bps in 1976-84, 34 in 1985-94, 18 in 1995-2004 and 11 in 2005-14, as volume grew. A manager starting in 2005 would face less than the full-sample capacity figure implies. Volume is in reported-series units, which I cannot check against contract volume.
- **The cost sits in thin markets.** At $1bn, commodities carry 80% of the impact (FX 14%), led by orange juice, platinum, soybean meal, oats and palladium. At $1bn, 17% of trades exceed 10% of a day's volume; a participation cap would bind there, and that is a different strategy from the one tested here.

### C5: slippage from the fill delay

**What we are doing.** The backtest trades at the month-end close. A real order placed after that close fills at the next open. The cost is the overnight gap in the direction of the trade.

**Inputs.** Daily Open and Close for the source held in each market-month; the signed trade (new position minus drifted prior position) from the 2 bps path; the trade date, which is the prior month's endpoint.

**How we process them.** `gap = Open(next day) / Close(trade date) - 1`; cost = signed trade x gap, so a purchase into a rising gap is a cost and a sale into one is a gain. The comparison is the same signed trade times the mean overnight gap on the other days of that month. Terminal liquidation is excluded (no later open). Source-switch legs take the net sign.

**Output.** Slippage in bps per unit traded, the comparison, a t-statistic on the monthly difference, by subperiod and by class.

In [17]:
pd.read_csv(TESTS / "c5_slippage.csv").set_index("group")

,coverage,slippage_bps_per_unit,other_days_bps_per_unit,excess_t,slippage_ann_bps
group,,,,,
all,1.0000,0.0755,0.7784,-0.6449,0.6500
1976-1984,1.0000,-0.7061,2.2434,-0.8496,-4.8625
1985-1994,1.0000,-1.4647,1.5880,-1.2926,-13.2699
1995-2004,1.0000,1.3937,-0.1803,0.8626,13.1730
2005-2014,1.0000,0.7971,-0.0586,0.7090,7.0617


In [18]:
pd.read_csv(TESTS / "c5_by_class.csv", index_col=0)

,slippage_bps_per_unit
AssetClass,
COMM,-1.5107
EQ,2.4920
FI,0.9820
FX,1.2893


- **No systematic cost from filling at the next open.** Over 1976-2014 the signed gap averages 0.08 bps per unit traded (0.65 bps a year), against 0.78 bps on ordinary days; the difference is not significant (t = -0.6). All trades have a next open.
- **The sign changes across eras.** The gap favours the trade by 0.7 and 1.5 bps per unit in 1976-84 and 1985-94 and costs 1.4 and 0.8 bps in 1995-2004 and 2005-14 (t about 0.9 and 0.7, so neither is reliable). The recent figures are a third to a half of the 2 bps assumption, which is worth watching but not a basis for adding a charge.
- **By class** equities (+2.5), FX (+1.3) and bonds (+1.0) lean adverse and commodities lean favourable (-1.5). Commodity opens can be thin or set by a different session, so I would not read these as a stable pattern.
- **The gap is noise more than cost.** It averages less than 1 bp per unit while individual gaps run to tens of bps. Delayed fills add tracking error against the backtest, which this test does not size.

### All together

Volatility-scaled spread at 2 bps (C3), impact with `Y` = 0.5 (C4) and realised fill slippage (C5), replacing the flat 2 bps. This is conservative: the 2 bps flat rate was meant to cover the spread, and here the spread is charged again alongside impact.

In [19]:
pd.read_csv(TESTS / "combined.csv")

,case,aum,mean_ann,sharpe,cagr
0,flat 2 bps,NaN,0.1107,1.1991,0.1118
1,spread + impact + slippage,10000000.0000,0.1080,1.1699,0.1089
2,spread + impact + slippage,100000000.0000,0.1025,1.1108,0.1028
3,spread + impact + slippage,1000000000.0000,0.0851,0.9176,0.0839
4,spread + impact + slippage,10000000000.0000,0.0300,0.3008,0.0253


## Summary and recommendations

**What we found.** (Gross return 11.2% a year, turnover 8.7 times NAV, TREND with the 10% band, 1976-2014, in sample, conditional on the March 1997 endpoint.)
- **C0:** the flat rate is not what limits the strategy. Net return reaches zero at 129 bps per unit traded, 65 times the 2 bps assumption. Sharpe falls by 0.1 only at 12.8 bps. Re-running the engine changes the first-order answer by at most 0.005 Sharpe, so repricing a fixed trade path is a fair way to test richer costs.
- **C3:** tying the spread to volatility adds about 4% to cost and changes Sharpe by 0.001. The strategy does not trade in the most volatile months.
- **C4:** impact is the cost that matters, and it depends on AUM. Sharpe falls by 0.1 near $133m and net return reaches zero near $19bn at `Y` = 0.5; the ranges across `Y` and the ADV scale are $17m-$267m and $2.4bn-$38bn. The cost is concentrated in thin commodity markets and in the early sample.
- **C5:** filling at the next open costs 0.08 bps per unit traded over the sample, not distinguishable from ordinary overnight gaps. The sign flips between eras.
- **Combined:** net Sharpe of 1.17 at $10m, 1.11 at $100m, 0.92 at $1bn and 0.30 at $10bn, against 1.20 at a flat 2 bps.

**Recommendations**
1. **Keep 2 bps as the headline for a small account (below about $100m) and always report it with the 129 bps breakeven and the Sharpe at 10 and 20 bps.** The earlier deck stopped at 10 bps; the stress now runs to 100 bps and the breakeven is 129.
2. **Do not replace the flat rate with a volatility-linked spread or add a fill-delay charge.** Neither moves Sharpe by more than 0.002 in this sample.
3. **State capacity as a range, not a number.** If the report quotes an AUM, use the C4 impact table at the chosen size and the $17m-$267m range for the Sharpe point, and say that point values are unverified.
4. **Before relying on C4, verify the point values and the volume units.** Supplying the exchange contract sizes (or a file of them) is the single input that would most tighten the capacity figure; the spread in the table comes mostly from this and from `Y`.
5. **If larger AUM matters, test a participation cap or an exclusion list for the thinnest commodity markets.** That changes positions, so it needs the engine, not first-order repricing, and it is a different experiment from this one.
6. **Leave the engine as is.** A richer cost enters as a charge on the trade path; C0 shows that letting the band and limits react changes little.

**Limits.** One in-sample path, so no test here carries a confidence interval beyond the t-statistics shown. C3, C4 and C5 are first order. Impact assumes the month's trade executes on one day. Volume is in reported-series units. The DCC-GARCH volatility is a model forecast, not an observed quote.